# Trial Drilldown

1つの `trial_id` の config / window / summary / no-signal / trade明細を確認します。`TRIAL_ID=None` の場合は指定metricの最上位trialを選びます。trade明細は `run_sweep --keep-trades all`（またはエラー調査用の `on-error`）で保存されたrunのみ表示できます。


In [ ]:
# Parameters
RUN_ID = "latest"
TRIAL_ID = None
RUNS_ROOT = "research/data/runs"
METRIC = "return_to_dd"


In [ ]:
import json
from IPython.display import Markdown, display
from research.src.store.trial_store import TrialStore
from research.src.store.views import format_table, rank

store = TrialStore(RUNS_ROOT)
run_id = store.resolve_run_id(RUN_ID)
rows = store.load_trials(run_id)
if TRIAL_ID is None:
    ranked = rank(rows, by=METRIC, top_k=1)
    if not ranked:
        raise RuntimeError("no successful trials found")
    trial = ranked[0]
else:
    trial = next(row for row in rows if row.get("trial_id") == TRIAL_ID)
display(Markdown(f"## Trial: `{trial['trial_id']}`"))
display(Markdown("```json\n" + json.dumps({
    "run_id": run_id,
    "model_id": trial.get("model_id"),
    "dataset_key": trial.get("dataset_key"),
    "window": trial.get("window"),
    "tags": trial.get("tags"),
    "error": trial.get("error"),
}, ensure_ascii=False, indent=2, default=str) + "\n```"))


In [ ]:
display(Markdown("## Summary"))
summary = trial.get("summary", {})
summary_rows = [{"metric": key, "value": value} for key, value in sorted(summary.items())]
display(Markdown("```\n" + format_table(summary_rows, ["metric", "value"]) + "\n```"))


In [ ]:
display(Markdown("## Config"))
display(Markdown("```json\n" + json.dumps(trial.get("config", {}), ensure_ascii=False, indent=2, default=str) + "\n```"))


In [ ]:
display(Markdown("## No-signal reasons"))
reasons = trial.get("no_signal_reason_counts", {}) or {}
reason_rows = [
    {"reason": key, "count": value}
    for key, value in sorted(reasons.items(), key=lambda item: item[1], reverse=True)[:20]
]
display(Markdown("```\n" + format_table(reason_rows, ["reason", "count"]) + "\n```"))


In [ ]:
display(Markdown("## Trades / asset curve"))
try:
    trades = store.load_trades(run_id, trial["trial_id"])
except FileNotFoundError:
    display(Markdown(
        "trade明細ファイルが見つかりません。"
        "`python -m research.scripts.run_sweep --keep-trades all ...` "
        "で実行したrunのみ、ここに trade 明細を表示できます。"
    ))
else:
    if not trades:
        display(Markdown("保存された trade 明細は0件です。"))
    else:
        trade_columns = [
            "entry_time",
            "exit_time",
            "exit_reason",
            "scaled_pnl_pct",
            "r_multiple",
            "position_size_multiplier",
            "holding_bars",
        ]
        display(Markdown("### Trade rows (first 50)"))
        display(Markdown("```\n" + format_table(trades[:50], trade_columns) + "\n```"))

        equity_rows = []
        equity_index = 100.0
        for index, trade in enumerate(trades, start=1):
            scaled_pnl_pct = trade.get("scaled_pnl_pct")
            if not isinstance(scaled_pnl_pct, (int, float)):
                continue
            equity_index *= 1 + (scaled_pnl_pct / 100)
            equity_rows.append({
                "trade_index": index,
                "equity_index": round(equity_index, 6),
                "scaled_pnl_pct": scaled_pnl_pct,
                "exit_reason": trade.get("exit_reason"),
            })
        if equity_rows:
            display(Markdown("### Equity index (initial=100)"))
            display(Markdown("```\n" + format_table(
                equity_rows,
                ["trade_index", "equity_index", "scaled_pnl_pct", "exit_reason"],
            ) + "\n```"))
